Решите задачу классификации трансфомером на [наборе данных](https://huggingface.co/imdb/datasets)

In [20]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

lr = 2e-5
batch_size = 32
epochs = 5
train_size = 1000
test_size = 100
seed = 42
max_length = 256

In [ ]:
train_ds, test_ds = load_dataset("imdb", split=["train", "test"])
ds = DatasetDict({
    "train": train_ds.shuffle(seed=seed).select(range(train_size)),
    "test": test_ds.shuffle(seed=seed).select(range(test_size)),
})
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch, max_length=256):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)

tokenized = ds.map(tokenize, batched=True)
tokenized = tokenized.rename_column("label", "labels")

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

args = TrainingArguments(
    output_dir="imdb-clf",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    dataloader_num_workers=4,
    num_train_epochs=epochs,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
train_ds[9]

{'text': "This is said to be a personal film for Peter Bogdonavitch. He based it on his life but changed things around to fit the characters, who are detectives. These detectives date beautiful models and have no problem getting them. Sounds more like a millionaire playboy filmmaker than a detective, doesn't it? This entire movie was written by Peter, and it shows how out of touch with real people he was. You're supposed to write what you know, and he did that, indeed. And leaves the audience bored and confused, and jealous, for that matter. This is a curio for people who want to see Dorothy Stratten, who was murdered right after filming. But Patti Hanson, who would, in real life, marry Keith Richards, was also a model, like Stratten, but is a lot better and has a more ample part. In fact, Stratten's part seemed forced; added. She doesn't have a lot to do with the story, which is pretty convoluted to begin with. All in all, every character in this film is somebody that very few people 

In [22]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds)),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

# progress bar by cursor
from transformers.trainer_callback import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)
trainer.add_callback(ProgressCallback())

trainer.train()

  0%|          | 0/160 [00:00<?, ?it/s]

/Users/andrey/Ground/25AIMEPhI/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': '0.6082', 'eval_accuracy': '0.7', 'eval_f1': '0.7', 'eval_runtime': '1.319', 'eval_samples_per_second': '75.82', 'eval_steps_per_second': '3.033', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

{'eval_loss': '0.2798', 'eval_accuracy': '0.9', 'eval_f1': '0.8958', 'eval_runtime': '1.548', 'eval_samples_per_second': '64.62', 'eval_steps_per_second': '2.585', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '216.1', 'train_samples_per_second': '23.13', 'train_steps_per_second': '0.74', 'train_loss': '0.3364', 'epoch': '5'}


There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=160, training_loss=0.336431884765625, metrics={'train_runtime': 216.1327, 'train_samples_per_second': 23.134, 'train_steps_per_second': 0.74, 'total_flos': 331168496640000.0, 'train_loss': 0.336431884765625, 'epoch': 5.0})

In [23]:
trainer.evaluate()

/Users/andrey/Ground/25AIMEPhI/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/4 [00:00<?, ?it/s]

{'eval_loss': 0.27688539028167725,
 'eval_accuracy': 0.9,
 'eval_f1': 0.8936170212765957,
 'eval_runtime': 1.282,
 'eval_samples_per_second': 78.005,
 'eval_steps_per_second': 3.12,
 'epoch': 5.0}